# 📦 ضغط الفيديو والصوت — Beta

1. ابعت ملف الفيديو أو الصوت في قناة **📦 ضغط الفيديو والصوت** على تيليجرام.
2. ارجع هنا واضغط **▶ تشغيل**.
3. النتيجة هترجع لنفس القناة تلقائيًا.

**إعادة نتيجة قديمة:** اعمل Reply عليها واكتب `1` لنفس الإعداد، `2` لأصغر حجم، `3` للصوت فقط، أو رقم الحجم مثل `80`.

> أول تشغيل فقط هيطلب بيانات تيليجرام، وبعدها الإعدادات تتحفظ في Google Drive الخاص بك.


In [ ]:
#@title ▶ تشغيل
#@markdown الوضع الافتراضي مناسب لمعظم الاستخدامات. غيّره فقط عند الحاجة.
#@markdown خانة **الحجم** اختيارية؛ استخدمها فقط مع «حجم فيديو محدد»، مثال: 80.
الوضع = "تلقائي — مناسب لمعظم الاستخدامات" #@param ["تلقائي — مناسب لمعظم الاستخدامات", "صوت صغير جدًا", "فيديو متوازن", "فيديو سريع", "أصغر حجم للفيديو", "حجم فيديو محدد"]
الحجم = "" #@param {type:"string"}

import os
import re
import subprocess
import urllib.request
import urllib.error
import json
import time

REPO = "abdullahsamirashour/gpt"
BRANCH = "beta"
ENGINE_REL = "telegram-smart-compressor/engine.py"
ENGINE_PATH = "/content/telegram_smart_compressor_engine.py"

PROFILE_MAP = {
    "تلقائي — مناسب لمعظم الاستخدامات": "SMART_AUTO",
    "صوت صغير جدًا": "AUDIO_TINY",
    "فيديو متوازن": "VIDEO_BALANCED",
    "فيديو سريع": "VIDEO_FAST",
    "أصغر حجم للفيديو": "VIDEO_SMALLEST",
    "حجم فيديو محدد": "VIDEO_TARGET_SIZE",
}

os.environ["TSC_PROFILE"] = PROFILE_MAP.get(الوضع, "SMART_AUTO")
os.environ["TSC_TARGET_SIZE_MB"] = str(الحجم or "").strip()
os.environ["TSC_UPDATE_CHANNEL"] = BRANCH

def latest_sha():
    p = subprocess.run(
        ["git", "ls-remote", f"https://github.com/{REPO}.git", f"refs/heads/{BRANCH}"],
        capture_output=True,
        text=True,
        timeout=30,
    )
    if p.returncode == 0 and p.stdout.strip():
        sha = p.stdout.strip().split()[0]
        if re.fullmatch(r"[0-9a-f]{40}", sha):
            return sha

    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/git/ref/heads/{BRANCH}?t={int(time.time())}",
        headers={
            "Accept": "application/vnd.github+json",
            "User-Agent": "Smart-Compressor-Colab",
            "Cache-Control": "no-cache",
        },
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))["object"]["sha"]

def download_engine(sha):
    url = f"https://raw.githubusercontent.com/{REPO}/{sha}/{ENGINE_REL}"
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Smart-Compressor-Colab", "Cache-Control": "no-cache"},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        return response.read().decode("utf-8")

try:
    print("🔄 جاري تحميل أحدث نسخة تجريبية...")
    sha = latest_sha()
    engine = download_engine(sha)
    if len(engine) < 5000:
        raise RuntimeError("engine-too-small")
    compiled = compile(engine, ENGINE_PATH, "exec")
    print(f"✅ تم تحميل المحرك — commit {sha[:10]}")
    exec(compiled, globals(), globals())
except urllib.error.HTTPError as exc:
    print(f"❌ E001 — تعذر الوصول إلى GitHub (HTTP {exc.code}). جرّب مرة أخرى.")
except urllib.error.URLError:
    print("❌ E001 — تعذر الاتصال بـ GitHub. تأكد من الإنترنت وجرب مرة أخرى.")
except SyntaxError:
    print("❌ E002 — النسخة المنشورة فيها خطأ برمجي. لا تغيّر أي إعدادات وأرسل كود E002.")
except Exception as exc:
    print("❌ E003 — تعذر بدء المحرك.")
    print(f"التفاصيل المختصرة: {type(exc).__name__}: {str(exc)[:180]}")
